<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/aya_vision_8b_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering Example (Aya Vision 8B)

This notebook demonstrates a Visual Question Answering (VQA) / Multiple-Choice Question Answering (MCQA) task using [Aya Vision 8B](https://huggingface.co/CohereForAI/aya-vision-8b), an open-weights multilingual vision-language model from Cohere Labs, loaded via the Hugging Face `transformers` library.

It mirrors the structure of the BLIP VQA notebook, but swaps in Aya Vision 8B as the model and adds multilingual (English / Teso / Swahili) MCQA evaluation sourced from a Google Sheet.

### Install Necessary Libraries

Aya Vision requires a `transformers` build that includes support for the `AyaVision` architecture (`AutoModelForImageTextToText`). We also install `Pillow` for image processing, `torch`, `accelerate` for device placement, and `datasets`/`pandas` for loading the Google Sheet.

In [ ]:
# Install transformers (with Aya Vision support), Pillow, torch, and other necessary libraries
# bitsandbytes is required for 4-bit quantized loading, which lets the 8B model
# fit entirely on a free-tier T4 GPU (16GB VRAM) without CPU/disk offload.
!pip install -qqq 'git+https://github.com/huggingface/transformers.git' Pillow torch accelerate datasets pandas bitsandbytes


### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from PIL import Image
import torch
import requests
import io


### Load Pre-trained Aya Vision Model and Processor

We'll use the `CohereForAI/aya-vision-8b` model, an 8-billion parameter multilingual vision-language model. Note: this model is gated on Hugging Face — you'll need to accept the license on the [model page](https://huggingface.co/CohereForAI/aya-vision-8b) and authenticate (e.g. `huggingface-cli login` or `notebook_login()`) before it will download.

In [ ]:
model_id = "CohereForAI/aya-vision-8b"

# 4-bit quantization: shrinks the model to ~5-6GB so it fits entirely on a
# T4's 16GB of VRAM. Without this, transformers silently offloads part of
# the model to CPU RAM (slow) or even disk (extremely slow) to make it fit,
# which can turn a few-second generation into a many-minute one.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

from huggingface_hub import notebook_login
notebook_login()

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)




### Model Evaluation

In a real-world scenario, evaluating a VQA model can be done in a few ways.

- **Qualitative Evaluation** (Manual Inspection): For a single example, we can simply look at the image, the question, and the model's answer to see if it makes sense.

- **Quantitative Evaluation** (Using Datasets): This is a more rigorous evaluation — you would typically use a VQA benchmark dataset. Such datasets contain thousands or millions of image-question pairs, each with multiple human-annotated answers. The common metrics are:

- **Accuracy**: The percentage of times the model's answer exactly matches one of the ground-truth answers.

- **VQA Score**: A more nuanced metric often used in VQA challenges. For each question, if the model's answer is present in the human answers, it gets a score that's min(human_answers_count / 3, 1). This accounts for cases where multiple correct answers exist.

**To perform quantitative evaluation, you would:**

Load a VQA dataset.
Iterate through the dataset, feeding each image-question pair to your model.
Collect the model's predictions.
Compare the predictions against the ground-truth answers using the chosen metric.


In [ ]:
# Let's try a real world VQA model, Aya Vision, on MCQA data

import requests
from PIL import Image
import io
import os
import random
import tempfile
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText


In [ ]:
# ============================================================
# STEP 1: Load Model and Processor
# ============================================================
print("Loading model and processor...")

model_id = "CohereForAI/aya-vision-8b"

# 4-bit quantization: shrinks the model to ~5-6GB so it fits entirely on a
# T4's 16GB of VRAM. Without this, transformers silently offloads part of
# the model to CPU RAM (slow) or even disk (extremely slow) to make it fit —
# which is what caused earlier runs to hang for many minutes per question.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

# Sanity check: confirm every part of the model landed on the GPU (cuda:0),
# not 'cpu' or 'disk'. If you see anything other than 0 / 'cuda:0' here,
# generation will be slow again.
print("Model device:", model.device)

print("Model loaded.\n")

# ============================================================
# Helper: run Aya Vision chat inference on a single PIL image + prompt
# ============================================================
# Aya Vision expects images via the chat template, referenced either by a
# remote URL or a local file path. Since our dataset yields PIL Image
# objects (streamed from the Google Sheet), we write each image to a
# temporary file before passing it to the processor.
_TMP_IMAGE_DIR = tempfile.mkdtemp(prefix="aya_vision_tmp_")

def generate_text(image, prompt, max_new_tokens=10):
    """
    Generate a text response from Aya Vision given a PIL image and a text prompt.

    Args:
        image (PIL.Image.Image): The input image.
        prompt (str): The text prompt to send with the image.
        max_new_tokens (int): Maximum number of new tokens to generate.

    Returns:
        str: The model's decoded text response.
    """
    tmp_path = os.path.join(_TMP_IMAGE_DIR, "frame.jpg")
    image.convert("RGB").save(tmp_path)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": tmp_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # NOTE: padding=True is intentionally omitted here. It's a no-op for
    # single (batch-size-1) examples like ours, and including it triggers a
    # harmless 'Kwargs passed to processor.__call__ have to be in
    # processor_kwargs...' warning in newer transformers versions.
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    decoded = processor.tokenizer.decode(
        gen_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return decoded.strip()


##Using Google Drive for Direct Image Links

If you prefer to use Google Drive to host your images, follow these steps to get a direct download link:

1.  **Upload your image(s)** to Google Drive.
2.  **Right-click** on the image file and select **'Share'**.
3.  In the sharing dialog, change the access from 'Restricted' to **'Anyone with the link'**.
4.  **Copy the shareable link**. It will look something like this:
    `https://drive.google.com/file/d/FILE_ID/view?usp=sharing`
5.  **Convert the link to a direct download link** by extracting the `FILE_ID` and formatting it as:
    `https://drive.google.com/uc?export=download&id=FILE_ID`

    **Example:**
    If your shareable link is `https://drive.google.com/file/d/1aBcDeFGhIjKlMnOpQrStUvWxYz/view?usp=sharing`,
    your direct download link will be `https://drive.google.com/uc?export=download&id=1aBcDeFGhIjKlMnOpQrStUvWxYz`

6.  **Update your Google Sheet** with these direct download links in your `image_filename` column.

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Features, Value, Image as ImageFeature

# 1. Configuration
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTs9PcGM6qO4MeGzL33BtqTDhFFt7Cq4YPXolFy-lpsdHfP-yBjL_EFvyC2Saoj_AJ9dm9VT9udAJgj/pub?output=csv"
IMAGE_DIR = "/content/images/"

def clean_image_path_or_url(filename):
    """Cleans up whitespace and handles absolute web URLs or local paths."""
    if not isinstance(filename, str):
        return filename
    filename = filename.strip()

    # If it is an internet URL, use it directly. If it is a filename, prepend the folder path.
    if filename.startswith(('http://', 'https://')):
        return filename
    return os.path.join(IMAGE_DIR, filename)

# 2. Extract Data from Google Sheet
print("Fetching Google Sheet...")
df = pd.read_csv(GOOGLE_SHEET_URL)
df.columns = df.columns.str.strip()  # Clean whitespace from headers

# Ensure an Index ID column is present
if 'ID' not in df.columns:
    df['ID'] = range(1, len(df) + 1)

# 3. Map Data Structures for Lazy Loading
if 'image_filename' in df.columns:
    # Set up final resource pointers
    df['image'] = df['image_filename'].apply(clean_image_path_or_url)
    df = df.drop(columns=['image_filename'])

    # Infer metadata schema types for the text columns
    features_dict = {
        col: Value(dtype='string') if dtype == 'object' else
             Value(dtype='int64') if 'int' in str(dtype) else
             Value(dtype='float64') if 'float' in str(dtype) else
             Value(dtype='bool') if 'bool' in str(dtype) else Value(dtype='string')
        for col, dtype in df.dtypes.items() if col != 'image'
    }
    features_dict['image'] = ImageFeature()  # Stream directly from path/URL string on-the-fly

    # Instantiate the stable dataset mapping
    print("Building Hugging Face lazy-loading structure...")
    custom_dataset_from_sheet = Dataset.from_pandas(df, features=Features(features_dict))

    samples = custom_dataset_from_sheet.select(range(len(custom_dataset_from_sheet)))
    print(f"✅ Loaded {len(samples)} samples successfully. System RAM footprint: ~0 MB.")
else:
    print("❌ Error: Column 'image_filename' was not found in your Google Sheet.")


After running this cell, `samples` will contain your data loaded from the Google Sheet, with the images processed accordingly. Remember to then run the following cell to use `custom_dataset_from_sheet` for evaluation.

Now, the `samples` variable contains your custom data, and the subsequent evaluation steps will use this data.

In [ ]:
# To use your custom dataset, comment out any other dataset-loading cell and use the following:
dataset = custom_dataset_from_sheet
samples = dataset.select(range(len(custom_dataset_from_sheet)))


In [ ]:
# ============================================================
# STEP 2: Display Sample Images
# ============================================================
print("Displaying sample images...")

# Group samples by a unique identifier for the image.
# For this task, we'll use 'Category' as a grouping key, assuming all samples
# within the same 'Category' are intended to share the same visual context.
grouped_images = {}
for i, sample in enumerate(samples):
    category = sample["Category"] # Use Category for grouping
    if category not in grouped_images:
        grouped_images[category] = {
            "image": sample["image"], # Store the actual image object
            "question_indices": []    # Store original 1-based question numbers
        }
    grouped_images[category]["question_indices"].append(i + 1)

# Determine the number of unique images to plot
num_unique_images = len(grouped_images)
# Adjust figure size dynamically based on the number of unique images
fig, axes = plt.subplots(1, num_unique_images, figsize=(5 * num_unique_images, 4))

# Ensure 'axes' is always an iterable (list) for consistent indexing, even if only one subplot
if num_unique_images == 1:
    axes = [axes]

for idx, (category, group_info) in enumerate(grouped_images.items()):
    image_to_display = group_info["image"]
    question_numbers = group_info["question_indices"]

    # Format the question numbers for the title (e.g., "Q1", "Q1-4")
    q_label = f"Q{question_numbers[0]}"
    if len(question_numbers) > 1:
        q_label += f"-{question_numbers[-1]}"

    # Create the full title including the aggregated question label and category
    full_title = f"{q_label}\n{category}"

    axes[idx].imshow(image_to_display)
    axes[idx].set_title(full_title, fontsize=8)
    axes[idx].axis("off")

plt.suptitle("CVQA Images (English/Teso/Swahili)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a prompt.
    Instructs the model to answer with only a letter A/B/C/D.
    """
    prompt = (
        "Look at the image carefully and answer the following "
        "multiple choice question by selecting only the letter "
        "A, B, C, or D.\n\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer with only the letter (A, B, C, or D):"
    return prompt

def extract_predicted_label(predicted_text, choices):
    """
    Extract the predicted label (A/B/C/D) from model output.

    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, check if model output
                matches any choice text directly.
    If no label is found, defaults to a random choice (A,B,C,D) to ensure a prediction is always made.
    """
    # Strategy 1 — look for letter
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label

    # Strategy 2 — match answer text
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label

    # Default to a random choice if no label is found
    import random
    return random.choice(["A", "B", "C", "D"])


### Extend Evaluation to Teso and Swahili Questions

Now, we'll extend the evaluation to also process questions in Teso and Swahili. This requires specifying the column names in your Google Sheet that contain the localized questions and their corresponding choices. The `evaluate_sea_mcqa` function will be updated to evaluate English, Teso, and Swahili questions for each sample.

First, define the Teso- and Swahili-related column names here:

In [ ]:
# Define column names for Teso questions and answers
TESO_QUESTION_COLUMN = "native_question"
TESO_CORRECT_ANSWER_COLUMN = "correct_native"
TESO_WRONG_ANSWER_COLUMNS = ["wrong_native_o1", "wrong_native_o2", "wrong_native_o3"]

print(f"Teso Question Column: {TESO_QUESTION_COLUMN}")
print(f"Teso Correct Answer Column: {TESO_CORRECT_ANSWER_COLUMN}")
print(f"Teso Wrong Answer Columns: {TESO_WRONG_ANSWER_COLUMNS}")

print("\n")

# Define column names for Swahili questions and answers
SWA_QUESTION_COLUMN = "swa_question"
SWA_CORRECT_ANSWER_COLUMN = "correct_swa"
SWA_WRONG_ANSWER_COLUMNS = ["wrong_swa_o1", "wrong_swa_o2", "wrong_swa_o3"]

print(f"Swahili Question Column: {SWA_QUESTION_COLUMN}")
print(f"Swahili Correct Answer Column: {SWA_CORRECT_ANSWER_COLUMN}")
print(f"Swahili Wrong Answer Columns: {SWA_WRONG_ANSWER_COLUMNS}")


Now, the `evaluate_sea_mcqa` function processes English, Teso, and Swahili questions for each sample using Aya Vision's chat template via `generate_text`. The results include a `language` field to distinguish between them.

In [ ]:
# ============================================================
# STEP 3: Run Evaluation (Trilingual, Aya Vision 8B)
# ============================================================
import time

def _get_row_with_retry(samples, idx, max_retries=4, base_delay=5):
    """
    Fetch samples[idx] with retries. This is important because accessing a
    row from a `datasets.Dataset` built on a lazy Image feature triggers the
    actual image download/decode right here — NOT later when you explicitly
    touch sample['image']. A slow or flaky image URL (e.g. Google Drive) can
    raise a network timeout (httpx.ReadTimeout) at this exact point, outside
    of any try/except placed later in the loop body. Without a retry wrapper
    around this specific access, a single bad image kills the entire run.
    """
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return samples[idx]
        except Exception as e:
            last_err = e
            wait = base_delay * attempt
            print(f"  [Row {idx+1}] fetch attempt {attempt}/{max_retries} failed "
                  f"({type(e).__name__}: {e}). Retrying in {wait}s...", flush=True)
            time.sleep(wait)
    print(f"  [Row {idx+1}] giving up after {max_retries} attempts "
          f"({type(last_err).__name__}: {last_err}). Skipping this row.", flush=True)
    return None

def evaluate_sea_mcqa(samples, teso_question_col, teso_correct_ans_col, teso_wrong_ans_cols,
                       swa_question_col, swa_correct_ans_col, swa_wrong_ans_cols):
    """
    Evaluate Aya Vision on SEA-MCQA for English, Teso, and Swahili questions.
    Accuracy = % of samples where model selects the correct option.
    Random chance baseline = 25% (4 choices).
    """
    predictions = []
    skipped_rows = []
    total_rows = len(samples)

    # image is loaded ONCE per row (not once per language) and passed in,
    # since sample['image'] is a lazy datasets.Image feature that re-fetches
    # from its source URL/path on every access. Re-fetching it 3x per row
    # (once per language) was the main hidden cause of slow runs, especially
    # with Google Drive-hosted images.
    def _evaluate_single_question_language(sample, image, question_col, correct_ans_col, wrong_ans_cols, language_tag):
        question    = sample[question_col]
        correct_ans = sample[correct_ans_col]
        wrong_1     = sample[wrong_ans_cols[0]]
        wrong_2     = sample[wrong_ans_cols[1]]
        wrong_3     = sample[wrong_ans_cols[2]]

        # Shuffle choices so correct answer isn't always in same position
        choices = [correct_ans, wrong_1, wrong_2, wrong_3]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        # Build prompt
        prompt = build_mcqa_prompt(question, choices)

        predicted_text = "ERROR"
        predicted_label = None
        is_correct = False

        try:
            # Run Aya Vision via its chat template
            predicted_text = generate_text(image, prompt, max_new_tokens=10)

            # Extract predicted label using both strategies
            predicted_label = extract_predicted_label(predicted_text, choices)

            # Check correctness
            is_correct = (predicted_label == correct_label)

        except Exception as e:
            print(f"  Error processing sample ID {sample['ID']} ({language_tag} question): {e}")

        return {
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "language":        language_tag,
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        }

    run_start = time.time()
    for i in range(total_rows):
        row_start = time.time()
        print(f"[Row {i+1}/{total_rows}] fetching row...", flush=True)

        # Fetch the row itself with retries — this is where the image
        # download/decode actually happens for a lazy datasets.Image feature.
        sample = _get_row_with_retry(samples, i)
        if sample is None:
            skipped_rows.append(i + 1)
            continue

        try:
            image = sample["image"].convert("RGB")
        except Exception as e:
            print(f"  Failed to convert image for row {i+1} (ID={sample.get('ID')}): {e}. Skipping.", flush=True)
            skipped_rows.append(i + 1)
            continue

        img_load_time = time.time() - row_start
        print(f"  Row {i+1} ID={sample['ID']} ({sample['Category']}) ready in {img_load_time:.1f}s. "
              f"Running English/Teso/Swahili questions...", flush=True)

        # Evaluate English question
        english_prediction = _evaluate_single_question_language(
            sample, image, "eng_question", "correct_en",
            ["wrong_en_o1", "wrong_en_o2", "wrong_en_o3"], "English"
        )
        predictions.append(english_prediction)

        # Evaluate Teso question
        teso_prediction = _evaluate_single_question_language(
            sample, image, teso_question_col, teso_correct_ans_col,
            teso_wrong_ans_cols, "Teso"
        )
        predictions.append(teso_prediction)

        # Evaluate Swahili question
        swa_prediction = _evaluate_single_question_language(
            sample, image, swa_question_col, swa_correct_ans_col,
            swa_wrong_ans_cols, "Swahili"
        )
        predictions.append(swa_prediction)

        row_time = time.time() - row_start
        elapsed = time.time() - run_start
        avg_per_row = elapsed / (i + 1)
        remaining = avg_per_row * (total_rows - (i + 1))
        print(f"  Row {i+1}/{total_rows} done in {row_time:.1f}s. "
              f"Elapsed: {elapsed/60:.1f} min. Est. remaining: {remaining/60:.1f} min.\n", flush=True)

    overall_correct = sum(p['is_correct'] for p in predictions)
    overall_accuracy = (overall_correct / len(predictions)) * 100 if predictions else 0

    if skipped_rows:
        print(f"\nWARNING: skipped {len(skipped_rows)} row(s) after repeated fetch failures: {skipped_rows}\n")

    return overall_accuracy, predictions

# Call the evaluation function
accuracy, detailed_predictions = evaluate_sea_mcqa(
    samples,
    TESO_QUESTION_COLUMN, TESO_CORRECT_ANSWER_COLUMN, TESO_WRONG_ANSWER_COLUMNS,
    SWA_QUESTION_COLUMN, SWA_CORRECT_ANSWER_COLUMN, SWA_WRONG_ANSWER_COLUMNS
)

print("Evaluation complete for English, Teso, and Swahili questions.")


### Print Results (Trilingual)

This section displays the detailed evaluation results for English, Teso, and Swahili questions.

In [ ]:
# ============================================================
# STEP 4: Print Results (Trilingual)
# ============================================================
print("\n" + "="*60)
print("     MCQA Evaluation Results — Aya Vision 8B (English, Teso, Swahili)")
print("="*60)

correct_english = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'English')
total_english = sum(1 for p in detailed_predictions if p['language'] == 'English')
accuracy_english = (correct_english / total_english) * 100 if total_english else 0

correct_teso = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Teso')
total_teso = sum(1 for p in detailed_predictions if p['language'] == 'Teso')
accuracy_teso = (correct_teso / total_teso) * 100 if total_teso else 0

correct_swa = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Swahili')
total_swa = sum(1 for p in detailed_predictions if p['language'] == 'Swahili')
accuracy_swa = (correct_swa / total_swa) * 100 if total_swa else 0

for i, pred in enumerate(detailed_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"\n[{status}] Q{pred['sample_id']} ({pred['language']}) [{pred['category']}]")
    print(f"  Question: {pred['question']}")
    for label, choice in pred["choices"].items():
        marker = " ← correct" if choice == pred["correct_answer"] else ""
        print(f"  {label}. {choice}{marker}")
    print(f"  Model output:    {pred['predicted_text']}")
    print(f"  Predicted label: {pred['predicted_label']}  "
          f"|  Correct label: {pred['correct_label']}")

print("\n" + "="*60)
print(f"Overall Accuracy:        {accuracy:.2f}% ({sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}) ")
print(f"English Accuracy:        {accuracy_english:.2f}% ({correct_english}/{total_english})")
print(f"Teso Accuracy:           {accuracy_teso:.2f}% ({correct_teso}/{total_teso})")
print(f"Swahili Accuracy:        {accuracy_swa:.2f}% ({correct_swa}/{total_swa})")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)


### Visualise Results (Trilingual)

This section visualises the evaluation results, indicating correctness for English, Teso, and Swahili questions.

In [ ]:
# ============================================================
# STEP 5: Visualise Results (Trilingual)
# ============================================================

# Create a figure with a grid of subplots for each sample, showing the image on top
# and English / Teso / Swahili results stacked underneath.

num_samples = len(samples)
fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 8))

for i, sample in enumerate(samples):
    # Find predictions for this sample ID
    sample_preds = [p for p in detailed_predictions if p['sample_id'] == sample['ID']]

    english_pred = next((p for p in sample_preds if p['language'] == 'English'), None)
    teso_pred = next((p for p in sample_preds if p['language'] == 'Teso'), None)
    swa_pred = next((p for p in sample_preds if p['language'] == 'Swahili'), None)

    # Display image in the top row
    axes[0, i].imshow(sample["image"])
    axes[0, i].set_title(f"Q{sample['ID']} [{sample['Category']}]", fontsize=9)
    axes[0, i].axis("off")

    # Build a stacked results string for English / Teso / Swahili
    lines = []
    for label, pred in (("English", english_pred), ("Teso", teso_pred), ("Swahili", swa_pred)):
        if pred:
            status = "✓" if pred["is_correct"] else "✗"
            lines.append((f"{label}: {status}  Pred: {pred['predicted_label']} | Correct: {pred['correct_label']}",
                          "green" if pred["is_correct"] else "red"))
        else:
            lines.append((f"{label}: No Data", "gray"))

    axes[1, i].axis("off")
    y_positions = [0.8, 0.5, 0.2]
    for (text, color), y in zip(lines, y_positions):
        axes[1, i].text(0.5, y, text, fontsize=8, color=color, ha='center', va='center',
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.suptitle(
    f"SEA-MCQA Results (English, Teso & Swahili) — Aya Vision 8B — Overall Accuracy: {accuracy:.1f}% "
    f"(Random baseline: 25%)",
    fontsize=12
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("\n" + "="*55)
print("     MCQA Evaluation Results (English/Teso/Swahili) — Aya Vision 8B")
print("="*55)

print(f"\nOverall Accuracy:        {accuracy:.2f}%")
print(f"Correct:                 "
      f"{sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*55)
